In [2]:
# Cross-image hallucination benchmark for LLaVA-OneVision-1.5-8B-Instruct.
# Runs 16 probes on test images, scores each answer, and writes schema v1.0 JSON
# (llava_ov_probe_results.json) for merging with SFTvsRLHF_multi.ipynb via compare_probe_results.py.

from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import torch

# ==================== Configuration ====================
# NOTEBOOK_DIR: working directory when the notebook is launched.
# IMAGE_DIR:    shared probe images (same set used by the SFT/RLHF notebook).
# MODEL_PATH:   local OneVision checkpoint (requires transformers==4.53.0 for valid logits).
# OUTPUT_JSON:  standardized benchmark output consumed by compare_probe_results.py.
NOTEBOOK_DIR = Path.cwd()
IMAGE_DIR = NOTEBOOK_DIR.parent / "test_images"
MODEL_PATH = Path("../../LLaVA-RLHF/models/LLaVA-OneVision-1.5-8B-Instruct").resolve()
OUTPUT_JSON = NOTEBOOK_DIR / "llava_ov_probe_results.json"

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Image dir:    {IMAGE_DIR}")
print(f"Model path:   {MODEL_PATH}")

Notebook dir: /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/natural_scenes
Image dir:    /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/mywork/test_images
Model path:   /data/workspaces/zwang829/workspace/algorithm/LLaVA_RLHF/LLaVA-RLHF/models/LLaVA-OneVision-1.5-8B-Instruct


In [3]:
# ==================== Model loading helpers ====================
# OneVision uses HuggingFace AutoModel + Qwen-VL chat template. transformers>=5.0 can
# load the checkpoint after patching missing APIs, but produces garbage logits — we
# hard-require 4.53.0 (the version the checkpoint was built for).


def apply_transformers_compat_patches() -> str:
    """Patch missing transformers>=5 APIs so the model can at least be imported.

    These patches are NOT sufficient for correct inference; they only unblock loading.

    Returns:
        Installed transformers version string.
    """
    import transformers

    version = transformers.__version__
    major = int(version.split(".", maxsplit=1)[0])

    if major >= 5:
        import torch.nn as nn
        import transformers.cache_utils as cache_utils
        import transformers.configuration_utils as configuration_utils
        import transformers.modeling_rope_utils as rope_utils
        from transformers.modeling_utils import PreTrainedModel

        # Backfill APIs removed/renamed in transformers 5.x.
        if not hasattr(cache_utils, "SlidingWindowCache"):
            cache_utils.SlidingWindowCache = cache_utils.StaticCache

        if not hasattr(configuration_utils, "layer_type_validation"):

            def _layer_type_validation(layer_types):
                """No-op validator stub for transformers>=5 compatibility."""
                return layer_types

            configuration_utils.layer_type_validation = _layer_type_validation

        if not hasattr(PreTrainedModel, "set_submodule"):

            def _set_submodule(self, target: str, module: nn.Module) -> None:
                """Replace a named submodule, mirroring the transformers 5.x API."""
                atoms = target.split(".")
                if len(atoms) == 1:
                    setattr(self, atoms[0], module)
                    return
                parent = self.get_submodule(".".join(atoms[:-1]))
                setattr(parent, atoms[-1], module)

            PreTrainedModel.set_submodule = _set_submodule

        if "default" not in rope_utils.ROPE_INIT_FUNCTIONS:

            def _compute_default_rope_parameters(config, device=None, seq_len=None, **kwargs):
                """Compute default RoPE frequencies for checkpoints missing the 'default' initializer."""
                base = config.rope_theta
                dim = getattr(config, "head_dim", None) or (
                    config.hidden_size // config.num_attention_heads
                )
                inv_freq = 1.0 / (
                    base
                    ** (
                        torch.arange(0, dim, 2, dtype=torch.int64)
                        .to(device=device, dtype=torch.float)
                        / dim
                    )
                )
                return inv_freq, 1.0

            rope_utils.ROPE_INIT_FUNCTIONS["default"] = _compute_default_rope_parameters

        warnings.warn(
            "transformers>=5.0 can load this checkpoint but produces invalid outputs. "
            "For working inference run: pip install 'transformers==4.53.0'",
            stacklevel=2,
        )

    return version


def load_config(model_path: str):
    """Load model config and ensure pad_token_id is set (required for generation).

    Args:
        model_path: Directory containing the OneVision checkpoint files.

    Returns:
        AutoConfig with text_config.pad_token_id populated.
    """
    from transformers import AutoConfig

    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    if not hasattr(config.text_config, "pad_token_id") or config.text_config.pad_token_id is None:
        with open(os.path.join(model_path, "generation_config.json")) as f:
            config.text_config.pad_token_id = json.load(f)["pad_token_id"]
    return config


def count_meta_parameters(model) -> int:
    """Count parameters still on the meta device (sign of incomplete GPU load / OOM).

    Args:
        model: Loaded HuggingFace model.

    Returns:
        Number of parameters whose device type is "meta".
    """
    return sum(1 for _, param in model.named_parameters() if param.device.type == "meta")


def build_load_kwargs(model_path: str, config, use_quantization: bool, transformers_major: int):
    """Build from_pretrained kwargs: bf16/fp16, device_map=auto, optional 4-bit fallback.

    Args:
        model_path: Checkpoint directory path.
        config: Pre-loaded AutoConfig.
        use_quantization: If True, attach a 4-bit BitsAndBytesConfig.
        transformers_major: Major version of transformers (controls dtype kwarg name).

    Returns:
        Keyword-argument dict for AutoModelForCausalLM.from_pretrained.
    """
    kwargs = {
        "pretrained_model_name_or_path": model_path,
        "config": config,
        "device_map": "auto",
        "trust_remote_code": True,
    }
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if use_quantization:
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    # transformers 5.x renamed torch_dtype -> dtype.
    if transformers_major >= 5:
        kwargs["dtype"] = compute_dtype
    else:
        kwargs["torch_dtype"] = compute_dtype
    return kwargs


def load_ov_model_and_processor(model_path: str):
    """Load OneVision model and processor; fall back to 4-bit if full-precision OOMs.

    Args:
        model_path: Directory containing the OneVision checkpoint files.

    Returns:
        Tuple of (model, processor) in eval mode, ready for inference.
    """
    from transformers import AutoModelForCausalLM, AutoProcessor

    transformers_version = apply_transformers_compat_patches()
    transformers_major = int(transformers_version.split(".", maxsplit=1)[0])
    if transformers_major >= 5:
        raise RuntimeError(
            "transformers>=5.0 is not supported for inference. "
            "Run: pip install 'transformers==4.53.0'"
        )

    config = load_config(model_path)
    dtype_name = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
    print(f"Loading LLaVA-OneVision model ({dtype_name})...")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            model = AutoModelForCausalLM.from_pretrained(
                **build_load_kwargs(model_path, config, use_quantization=False, transformers_major=transformers_major)
            )
            if count_meta_parameters(model):
                raise torch.cuda.OutOfMemoryError("weights left on meta device")
        except torch.cuda.OutOfMemoryError:
            print("[WARN] Insufficient VRAM; retrying with 4-bit quantization...")
            torch.cuda.empty_cache()
            model = AutoModelForCausalLM.from_pretrained(
                **build_load_kwargs(model_path, config, use_quantization=True, transformers_major=transformers_major)
            )

    model.eval()
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    print("[OK] Model and processor ready")
    return model, processor

In [4]:
# ==================== Probe definitions & scoring (schema v1.0) ====================
# 16 probes across 6 images test existence / count / relation / attribute / open answers.
# Probe types:
#   existence  — yes/no about an object that is or is not in the image
#   count      — numeric answer
#   relation   — spatial or depth relation (left/right, front/behind)
#   attribute  — color or other property
#   open       — free-form description; scored by absence of invented natural-scene objects
#
# Each probe has a ground-truth (gt) and a short note explaining the intended trap.
# Output format matches SFTvsRLHF_multi.ipynb so compare_probe_results.py can merge runs.

PROBES = [
    {
        "image": "test.jpg",
        "id": "T1",
        "type": "existence",
        "question": "Is there a keyboard in this image? Answer yes or no.",
        "gt": "no",
        "note": "Keyboard is plausible in an office scene but is not present"
    },
    {
        "image": "test.jpg",
        "id": "T2",
        "type": "count",
        "question": "How many tissue boxes are in this image? Answer with a number.",
        "gt": "1",
        "note": "Exactly one tissue box; SFT+ answered correctly, RLHF once regressed to 2"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C1",
        "type": "existence",
        "question": "Is there a coffee machine in this image? Answer yes or no.",
        "gt": "no",
        "note": "Coffee machine is common in kitchens but is not in the image"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C2",
        "type": "existence",
        "question": "Is there a banana in the bowl? Answer yes or no.",
        "gt": "no",
        "note": "Fruit bowl contains only apples and pears"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C3",
        "type": "relation",
        "question": "Is the bowl of fruit to the left or right of the sink? Answer left or right.",
        "gt": "left",
        "note": "Fruit bowl is to the left of the sink"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E1",
        "type": "existence",
        "question": "Is there a keyboard on the desk? Answer yes or no.",
        "gt": "no",
        "note": "Keyboard is typical desk clutter but is not present"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E2",
        "type": "existence",
        "question": "Is there a computer monitor in this image? Answer yes or no.",
        "gt": "no",
        "note": "Monitor is common on desks but is not in the image"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E3",
        "type": "count",
        "question": "How many cups are in this image? Answer with a number.",
        "gt": "1",
        "note": "Only one gray mug on the desk"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K1",
        "type": "existence",
        "question": "Is there a cat in this image? Answer yes or no.",
        "gt": "no",
        "note": "Drawing shows a dog, not a cat"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K2",
        "type": "count",
        "question": "How many people are in this image? Answer with a number.",
        "gt": "5",
        "note": "Five stick-figure people"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K3",
        "type": "attribute",
        "question": "What color is the dress of the person on the far left? Answer with one color.",
        "gt": "yellow",
        "note": "Leftmost figure wears a yellow dress"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O1",
        "type": "relation",
        "question": "Is the paper boat in front of or behind the pen holder? Answer front or behind.",
        "gt": "front",
        "note": "Paper boat is in front of the pen holder"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O2",
        "type": "relation",
        "question": "Is the pink pen in front of or behind the paper boat? Answer front or behind.",
        "gt": "front",
        "note": "Pink pen is in front of the paper boat"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O3",
        "type": "existence",
        "question": "Is there a yellow sticky note on the wall? Answer yes or no.",
        "gt": "yes",
        "note": "Yellow sticky note is visible on the background wall"
    },
    {
        "image": "abstract_painting.jpg",
        "id": "A1",
        "type": "existence",
        "question": "Is there a tree in this image? Answer yes or no.",
        "gt": "no",
        "note": "Pure abstract painting with no tree"
    },
    {
        "image": "abstract_painting.jpg",
        "id": "A2",
        "type": "open",
        "question": "Describe this image in detail.",
        "gt": "abstract",
        "note": "Open-ended: check whether the model invents natural-scene objects"
    }
]

import json

SCHEMA_VERSION = "1.0"
BENCHMARK_NAME = "cross_image_hallucination_probe"

# Words that should NOT appear in open-ended answers on abstract/non-scene images.
NATURAL_OBJECTS = [
    "mountain", "lake", "tree", "forest", "person", "people",
    "man", "woman", "dog", "cat", "house", "building", "car", "sky", "river",
]


def judge_correctness(pred, gt, qtype):
    """Score a model answer against ground truth using loose heuristics.

    Args:
        pred: Raw model response text.
        gt: Ground-truth answer string from the probe definition.
        qtype: Probe type — existence, count, relation, attribute, or open.

    Returns:
        True if the answer passes the probe's correctness check.
    """
    pred_lower = pred.lower().strip(".")
    gt_lower = gt.lower().strip()
    if qtype == "open":
        return not any(obj in pred_lower for obj in NATURAL_OBJECTS)
    if gt_lower in ["yes", "no"]:
        if gt_lower == "yes":
            return "yes" in pred_lower and "no" not in pred_lower
        return "no" in pred_lower and "yes" not in pred_lower
    return gt_lower in pred_lower


def _verdict_label(correct, qtype):
    """Map a boolean correctness flag to a JSON verdict string.

    Args:
        correct: Whether judge_correctness returned True.
        qtype: Probe type; open probes use hallucination-specific labels.

    Returns:
        Verdict label, e.g. "correct", "wrong", or "natural_object_hallucination".
    """
    if qtype == "open":
        return "no_natural_object_hallucination" if correct else "natural_object_hallucination"
    return "correct" if correct else "wrong"


def build_benchmark_output(run_id, models, probes, answers_by_model, pairwise=None):
    """Build the standardized schema v1.0 benchmark JSON payload.

    Args:
        run_id: Identifier for this benchmark run (e.g. "llava_onevision").
        models: List of dicts with key, name, and path for each evaluated model.
        probes: Probe definitions (the PROBES list).
        answers_by_model: Dict mapping model key -> {probe_id: answer_text}.
        pairwise: Optional two model keys for head-to-head summary stats.

    Returns:
        Dict ready to serialize as cross-image probe results JSON.
    """
    model_keys = [m["key"] for m in models]
    per_probe = []
    for probe in probes:
        answers, correct, verdict = {}, {}, {}
        for key in model_keys:
            ans = answers_by_model[key].get(probe["id"], "")
            ok = judge_correctness(ans, probe["gt"], probe["type"])
            answers[key] = ans
            correct[key] = ok
            verdict[key] = _verdict_label(ok, probe["type"])
        per_probe.append({**probe, "answers": answers, "correct": correct, "verdict": verdict})

    by_model = {}
    for key in model_keys:
        c = sum(1 for r in per_probe if r["correct"][key])
        t = len(per_probe)
        by_type = {}
        open_halluc = open_ok = 0
        for r in per_probe:
            q = r["type"]
            by_type.setdefault(q, {"correct": 0, "wrong": 0})
            if r["correct"][key]:
                by_type[q]["correct"] += 1
            else:
                by_type[q]["wrong"] += 1
            if q == "open":
                if r["correct"][key]:
                    open_ok += 1
                else:
                    open_halluc += 1
        by_model[key] = {
            "correct": c, "wrong": t - c, "accuracy": round(c / max(t, 1), 4),
            "open_halluc": open_halluc, "open_no_halluc": open_ok, "by_type": by_type,
        }

    summary = {"total": len(per_probe), "by_model": by_model, "pairwise": None}
    if pairwise and len(pairwise) == 2:
        left, right = pairwise
        pw = {"both_correct": 0, "both_wrong": 0, f"{left}_better": 0, f"{right}_better": 0}
        for r in per_probe:
            if r["type"] == "open":
                continue
            l_ok, r_ok = r["correct"][left], r["correct"][right]
            if l_ok and r_ok:
                pw["both_correct"] += 1
            elif not l_ok and not r_ok:
                pw["both_wrong"] += 1
            elif l_ok:
                pw[f"{left}_better"] += 1
            else:
                pw[f"{right}_better"] += 1
        summary["pairwise"] = pw

    return {
        "schema_version": SCHEMA_VERSION,
        "benchmark": BENCHMARK_NAME,
        "run_id": run_id,
        "models": models,
        "probes": probes,
        "per_probe": per_probe,
        "summary": summary,
    }


def save_benchmark_output(output, path):
    """Write benchmark output dict to a JSON file with UTF-8 encoding.

    Args:
        output: Schema v1.0 payload from build_benchmark_output.
        path: Destination file path.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)


In [5]:
# ==================== Inference ====================
# OneVision expects Qwen-VL style chat messages (image + text). Greedy decoding;
# open-ended "Describe..." probes get a longer token budget.

from qwen_vl_utils import process_vision_info


def generate_answer_ov(model, processor, image_path: str, question: str) -> str:
    """Run one (image, question) pair through the OneVision chat template.

    Args:
        model: Loaded LLaVA-OneVision causal LM.
        processor: Matching AutoProcessor with chat template support.
        image_path: Path to the probe image file.
        question: Probe question text.

    Returns:
        Decoded assistant reply (prompt tokens stripped).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    max_new_tokens = 128 if question.startswith("Describe") else 32
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only the newly generated tokens (strip the prompt).
    trimmed = [
        out_ids[len(in_ids) :]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


In [6]:
# ==================== Load model ====================
# Loads bf16 (or fp16) on GPU; automatically retries with 4-bit if VRAM is insufficient.
model, processor = load_ov_model_and_processor(MODEL_PATH)

/data/workspaces/zwang829/conda_envs/llava-ov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading LLaVA-OneVision model (bf16)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.59it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


[OK] Model and processor ready


In [7]:
# ==================== Run all probes ====================
# Iterate over PROBES, run inference per image, and store raw answers in results_ov.
print("=" * 80)
print("Running LLaVA-OneVision on all probes...")
print("=" * 80)

results_ov = {}

for p in PROBES:
    img_path = IMAGE_DIR / p["image"]
    if not img_path.is_file():
        print(f"[SKIP] {img_path} not found")
        continue

    ans = generate_answer_ov(model, processor, str(img_path), p["question"])
    results_ov[p["id"]] = ans

    correct = judge_correctness(ans, p["gt"], p["type"])
    status = "OK" if correct else "WRONG"
    print(f"[OV][{p['id']}] {p['image'][:18]:<18} | {status:<5} | {ans[:60]}")

Running LLaVA-OneVision on all probes...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][T1] test.jpg           | OK    | No


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][T2] test.jpg           | OK    | 1


...

[OV][O1] objects_overlappin | OK    | front
[OV][O2] objects_overlappin | OK    | front


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][O3] objects_overlappin | OK    | yes


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OV][A1] abstract_painting. | OK    | No
[OV][A2] abstract_painting. | OK    | The image is an abstract painting characterized by vibrant, 


In [8]:
# ==================== Summary table & JSON export ====================
# Build schema v1.0 output, print a human-readable table, then save for compare_probe_results.py.
benchmark_output = build_benchmark_output(
    run_id="llava_onevision",
    models=[
        {
            "key": "llava_ov",
            "name": "LLaVA-OneVision-1.5-8B-Instruct",
            "path": str(MODEL_PATH),
        }
    ],
    probes=PROBES,
    answers_by_model={"llava_ov": results_ov},
)
per_probe = benchmark_output["per_probe"]
summary = benchmark_output["summary"]
ov_stats = summary["by_model"]["llava_ov"]

print("\n" + "=" * 100)
print("LLaVA-OneVision CROSS-IMAGE HALLUCINATION PROBE RESULTS")
print("=" * 100)
print(f"{'ID':<4} {'Image':<18} {'Type':<12} {'GT':<10} {'Answer':<35} {'Verdict'}")
print("-" * 100)

for row in per_probe:
    ans = row["answers"]["llava_ov"]
    verdict = row["verdict"]["llava_ov"].replace("_", " ")
    print(
        f"{row['id']:<4} {row['image'][:17]:<18} {row['type']:<12} "
        f"{row['gt']:<10} {ans[:34]:<35} {verdict}"
    )

print("-" * 100)
print(
    f"\n[Open-ended Abstract] hallucination count: {ov_stats['open_halluc']}, "
    f"no-hallucination count: {ov_stats['open_no_halluc']}"
)
print(
    f"\nSummary: correct={ov_stats['correct']}, wrong={ov_stats['wrong']}, "
    f"accuracy={ov_stats['accuracy']:.1%}"
)
print("=" * 100)

save_benchmark_output(benchmark_output, OUTPUT_JSON)
print(f"\n[OK] Standardized results saved to {OUTPUT_JSON}")


LLaVA-OneVision CROSS-IMAGE HALLUCINATION PROBE RESULTS
ID   Image              Type         GT         Answer                              Verdict
----------------------------------------------------------------------------------------------------
T1   test.jpg           existence    no         No                                  correct
T2   test.jpg           count        1          1                                   correct
C1   clean_counter.jpg  existence    no         No, there is no coffee machine vis  correct
C2   clean_counter.jpg  existence    no         No, there is no banana in the bowl  correct
C3   clean_counter.jpg  relation     left       left                                correct
E1   empty_desk.jpg     existence    no         No, there is no keyboard visible o  correct
E2   empty_desk.jpg     existence    no         No                                  correct
E3   empty_desk.jpg     count        1          1                                   correct
K1   kids_draw